# LLM with Key Value Caching


# Implementing Transformer Architecture: A Step-by-Step Guide


- Key sections:
- 3.1: Encoder and Decoder Stacks
- 3.2: Attention Mechanism
- 3.3: Position-wise Feed-Forward Networks
- 3.4: Embeddings and Softmax
- 3.5: Positional Encoding
- 5.4: Regularization (dropout strategy)

## Implementation Strategy
Breaking down the architecture into manageable pieces and gradually adding complexity:

1. Start with foundational components:
    - Embedding + Positional Encoding
    - Single-head self-attention

2. Build up attention mechanism:
- Extend to multi-head attention
- Add cross-attention capability
- Implement attention masking

3. Construct larger components:
- Encoder (self-attention + FFN)
- Decoder (masked self-attention + cross-attention + FFN)

4. Combine into final architecture:
- Encoder-Decoder stack
- Full Transformer with input/output layers

## Development Tips
1. Visualization and Planning:
- Draw out tensor dimensions on paper
- Sketch attention patterns and masks
- Map each component back to paper equations
- This helps catch dimension mismatches early!

2. Dimension Cheat Sheet:
- Input tokens: [batch_size, seq_len]
- Embeddings: [batch_size, seq_len, d_model]
- Attention matrices: [batch_size, num_heads, seq_len, seq_len]
- FFN hidden layer: [batch_size, seq_len, d_ff]
- Output logits: [batch_size, seq_len, vocab_size]

3. Common Pitfalls:
- Forgetting to scale dot products by √d_k
- Applying mask too early or too late
- Incorrect mask dimensions or application
- Missing residual connections
- Wrong order of layer norm and dropout
- Tensor dimension mismatches in attention
- Not handling padding properly

4. Performance Considerations:
- Memory usage scales with sequence length squared
- Attention computation is O(n²) with sequence length
- Balance between d_model and num_heads
- Trade-off between model size and batch size

## Testing Strategy
- Test each component independently
- Verify shape preservation
- Check attention patterns
- Confirm mask effectiveness
- Validate gradient flow
- Monitor numerical stability

Remember: The key to successfully implementing the Transformer is understanding how each piece fits together and maintaining clear dimension tracking throughout the implementation

In [1]:
import os

import pandas as pd

# Sets the high watermark ratio to 0.0, completely disabling the upper memory allocation limits
os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.0"

In [2]:
os.makedirs("checkpoints", exist_ok=True)
checkpoint_dir = "checkpoints/"

In [3]:
import gc
import torch


def clean_memory_cache():

    if torch.mps.is_available():
        print(
            f"Before Clearing, Available memory: {torch.mps.driver_allocated_memory() / 1024 / 1024:.2f} MB"
        )
        gc.collect()
        torch.mps.empty_cache()
    elif torch.cuda.is_available():
        print(
            f"Before Clearing, Available memory: {torch.cuda.memory_allocated() / 1024 / 1024:.2f} MB"
        )
        gc.collect()
        torch.cuda.empty_cache()
    else:
        gc.collect()
        return

## Transformer and Vision Transformer

In [4]:
from typing import Optional, Tuple, Any, List
import torch
import torch.nn as nn
from dataclasses import dataclass

import numpy as np
import torch
import torchvision
from torch import nn, optim
from torchvision import transforms
import sklearn
from sklearn.metrics import confusion_matrix
import tqdm
import copy
from torch.utils.data import DataLoader, Subset
import torch.nn.functional as F

torch.autograd.set_detect_anomaly(True)
device = (
    torch.accelerator.current_accelerator().type
    if torch.accelerator.is_available()
    else "cpu"
)
import os

num_workers = min(2, os.cpu_count())
import logging

# Configure logging to write directly to a file
logging.basicConfig(
    filename="vit.log",
    filemode="a",  # 'a' appends to the file, 'w' overwrites it on every run
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)


@dataclass(init=True)
class TrainResult:
    """
    A collection containing everything we need to know about the training results
    """

    train_losses: List[float]
    train_accs: List[float]
    val_accs: List[float]
    val_losses: List[float]


result = TrainResult(train_losses=[], train_accs=[], val_accs=[], val_losses=[])


class AverageMeter:
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0.0
        self.avg = 0.0
        self.sum = 0.0
        self.count = 0.0

    def update(self, val: float, n: int = 1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count if self.count > 0 else 0.0

    def calculate(self) -> float:
        return self.avg


def print_variance(name: str, data: torch.Tensor):
    # Compute variance across features/neurons and average across the batch
    neuron_variance = torch.mean(torch.var(data.detach().float(), dim=-1))
    print(f"{name}: Variance = {neuron_variance.item():.6f}")


import torch
import torch.nn as nn


class OptimizedAttentionLayer(nn.Module):
    def __init__(self, dim, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads

        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)
        self.out_proj = nn.Linear(dim, dim)

    def forward(self, x):
        batch_size, seq_len, dim = x.shape

        # Project and reshape for multi-head attention
        q = (
            self.q_proj(x)
            .view(batch_size, seq_len, self.num_heads, self.head_dim)
            .transpose(1, 2)
        )
        k = (
            self.k_proj(x)
            .view(batch_size, seq_len, self.num_heads, self.head_dim)
            .transpose(1, 2)
        )
        v = (
            self.v_proj(x)
            .view(batch_size, seq_len, self.num_heads, self.head_dim)
            .transpose(1, 2)
        )

        # --- THE MODERN OPTIMIZATION ---
        # Instead of manual matmul + scale + masked_fill + softmax + bmm,
        # use PyTorch's native SDPA scaled dot product attention. It automatically fuses these kernels
        # and triggers FlashAttention under the hood if hardware allows!
        out = F.scaled_dot_product_attention(q, k, v, is_causal=False)

        # Reshape back
        out = out.transpose(1, 2).contiguous().view(batch_size, seq_len, dim)
        return self.out_proj(out)


class MultiHeadedAttention(nn.Module):
    def __init__(self, dim: int, n_hidden: int, num_heads: int):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.n_hidden = n_hidden
        self.scale = n_hidden**-0.5

        self.qkv_projection = nn.Linear(dim, num_heads * n_hidden * 3, bias=False)
        self.W0 = nn.Linear(num_heads * n_hidden, dim)

    def forward(
        self,
        x: torch.Tensor,
        attn_mask: Optional[torch.Tensor] = None,
        layer_past: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
        use_cache: bool = False,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """


        Args:
            x: B,T,_ T=1 (during cached generation)
            attn_mask: mask
            layer_past:
            use_cache:

        Returns:

        """
        B, T, _ = x.shape

        # Shape: (B, T, 3 * num_heads * n_hidden) -> (B, num_heads, T, 3 * n_hidden)
        qkv = (
            self.qkv_projection(x)
            .reshape(B, T, self.num_heads, 3 * self.n_hidden)
            .transpose(1, 2)
        )
        # todo implement caching here for llm

        q, k, v = qkv.chunk(3, dim=-1)

        if layer_past is not None:
            past_k, past_v = layer_past
            # Concatenate past keys/values with the new token's key/value along the sequence dimension (T)
            k = torch.cat([past_k, k], dim=-2)
            v = torch.cat([past_v, v], dim=-2)
        # Save the current k and v for the next generation step
        present = (k, v) if use_cache else None

        # --- MODERNIZED ATTENTION MATH ---
        # SDPA automatically applies FlashAttention if available, skipping the manual score/softmax steps.
        # Note: If T=1 (during cached generation), causal masking is not needed because it is only looking backwards.
        is_causal = (attn_mask is None) and (T > 1) and (layer_past is None)

        context = F.scaled_dot_product_attention(
            q, k, v, attn_mask=attn_mask, is_causal=is_causal
        )

        # scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        #
        # if attn_mask is not None:
        #     if attn_mask.dim() == 3:
        #         attn_mask = attn_mask.unsqueeze(1)
        #     scores = scores.masked_fill(attn_mask == 0, float("-inf"))
        #
        # attn_weights = torch.softmax(scores, dim=-1)

        # context = torch.matmul(attn_weights, v)
        # context = context.transpose(1, 2).reshape(B, T, self.num_heads * self.n_hidden)
        # Add .contiguous() before reshaping the context matrix
        context = (
            context.transpose(1, 2)
            .contiguous()
            .view(B, T, self.num_heads * self.n_hidden)
        )
        output = self.W0(context)
        return output, present


class AttentionResidual(nn.Module):
    def __init__(self, dim: int, attn_dim: int, mlp_dim: int, num_heads: int):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)

        self.attn = MultiHeadedAttention(dim, attn_dim, num_heads)
        # self.attn = OptimizedAttentionLayer(dim,attn_dim, num_heads)
        # LayerNorm applied inside the FFN sequence only
        self.norm2 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(
            # nn.LayerNorm(dim),
            nn.Linear(dim, mlp_dim),
            nn.GELU(),
            nn.Linear(mlp_dim, dim),
        )

    def forward(
        self,
        x: torch.Tensor,
        attn_mask: Optional[torch.Tensor] = None,
        layer_past: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
        is_causal: bool = False,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        # Attention block with residual connection (no norm)
        attn_out, alphas = self.attn(
            x=self.norm1(x),
            attn_mask=attn_mask,
            layer_past=layer_past,
            is_causal=is_causal,
        )
        x = x + attn_out

        # FFN block with residual connection (norm is first layer inside self.ffn)
        x = x + self.ffn(self.norm2(x))
        return x, alphas


class Transformer(nn.Module):
    def __init__(
        self, dim: int, attn_dim: int, mlp_dim: int, num_heads: int, num_layers: int
    ):
        super().__init__()
        self.layers = nn.ModuleList(
            [
                AttentionResidual(dim, attn_dim, mlp_dim, num_heads)
                for _ in range(num_layers)
            ]
        )

    def forward(
        self,
        x: torch.Tensor,
        attn_mask: Optional[torch.Tensor] = None,
        return_attn: bool = False,
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        collected_attns = []

        for layer in self.layers:
            x, alphas = layer(x, attn_mask=attn_mask)
            if return_attn:
                collected_attns.append(alphas)

        if return_attn:
            return x, torch.stack(collected_attns, dim=1)
        return x, None


class PatchEmbed(nn.Module):
    def __init__(self, img_size: int, patch_size: int, nin: int, nout: int):
        super().__init__()
        assert (
            img_size % patch_size == 0
        ), "Image dimensions must be divisible by patch size."

        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2

        self.proj = nn.Conv2d(
            in_channels=nin,
            out_channels=nout,
            kernel_size=patch_size,
            stride=patch_size,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Output: (B, num_patches, nout) in float
        return self.proj(x).flatten(2).transpose(1, 2)


class VisionTransformer(nn.Module):
    def __init__(
        self,
        n_channels: int,
        nout: int,
        img_size: int,
        patch_size: int,
        dim: int,
        attn_dim: int,
        mlp_dim: int,
        num_heads: int,
        num_layers: int,
        num_global_tokens: int = 3,  # Number of global/CLS tokens
    ):
        super().__init__()
        self.num_global_tokens = num_global_tokens

        self.patch_embed = PatchEmbed(
            img_size=img_size, patch_size=patch_size, nin=n_channels, nout=dim
        )
        num_patches = self.patch_embed.num_patches

        # Learnable global/CLS tokens of shape (1, num_global_tokens, dim)

        self.cls_tokens = nn.Parameter(torch.zeros(1, num_global_tokens, dim))

        # Position embeddings covering both global tokens and image patches
        total_seq_len = num_global_tokens + num_patches
        self.pos_embed = nn.Parameter(torch.zeros(1, total_seq_len, dim))

        nn.init.trunc_normal_(self.cls_tokens, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        self.transformer = Transformer(
            dim=dim,
            attn_dim=attn_dim,
            mlp_dim=mlp_dim,
            num_heads=num_heads,
            num_layers=num_layers,
        )

        # Classification / Projection Head
        self.head = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, nout),
        )

    def forward(
        self, img: torch.Tensor, return_attn: bool = False
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        B = img.shape[0]

        # 1. Patch embeddings: (B, num_patches, dim)
        embs = self.patch_embed(img)

        # 2. Expand global tokens across batch: (B, num_global_tokens, dim)
        cls_tokens = self.cls_tokens.expand(B, -1, -1)

        # 3. Concatenate global tokens with patch embeddings: (B, num_global_tokens + num_patches, dim)
        x = torch.cat((cls_tokens, embs), dim=1)

        # 4. Add position embeddings
        x = x + self.pos_embed

        # 5. Transformer forward pass
        x, alphas = self.transformer(
            x, attn_mask=None, return_attn=return_attn
        )  # is_causal evaluates to True

        # 6. Extract representations of all global tokens: (B, num_global_tokens, dim)
        global_repr = x[:, : self.num_global_tokens]

        # Aggregate global tokens:
        # - Option A: Use the primary token (index 0) if acting as a single CLS token with register tokens
        # out = self.head(global_repr[:, 0])
        #
        # - Option B: Mean-pool across all global tokens
        out = self.head(global_repr.mean(dim=1))  # (B, nout)

        return out, alphas


# evaluate the model
@torch.compile
def evaluate_cifar_model(model, criterion, val_loader):
    is_train = model.training
    model.eval()
    with torch.no_grad():
        loss_meter, acc_meter = AverageMeter(), AverageMeter()
        for img, labels in val_loader:
            # move all img, labels to device (cuda)
            img = img.to(device)
            labels = labels.to(device)
            outputs, _ = model(img)
            loss_meter.update(criterion(outputs, labels).item(), len(img))
            acc = (outputs.argmax(-1) == labels).float().mean().item()
            acc_meter.update(acc, len(img))
    model.train(is_train)
    return loss_meter.calculate(), acc_meter.calculate()


result = TrainResult(train_losses=[], train_accs=[], val_losses=[], val_accs=[])


def main():
    model = VisionTransformer(
        n_channels=3,
        nout=10,
        img_size=32,
        patch_size=4,
        dim=128,
        attn_dim=64,
        mlp_dim=128,
        num_heads=3,
        num_layers=6,
    ).to(device)
    # print(model)
    MEAN = [0.4914, 0.4822, 0.4465]
    STD = [0.247, 0.2435, 0.2616]
    img_transform = transforms.Compose(
        [transforms.ToTensor(), transforms.Normalize(mean=MEAN, std=STD)]
    )
    inv_transform = transforms.Compose(
        [
            transforms.Normalize(mean=[0.0, 0.0, 0.0], std=1 / np.array(STD)),
            transforms.Normalize(mean=-np.array(MEAN), std=[1.0, 1.0, 1.0]),
            transforms.ToPILImage(),
        ]
    )
    train_dataset = torchvision.datasets.CIFAR10(
        train=True, root="data", transform=img_transform, download=True
    )
    val_dataset = torchvision.datasets.CIFAR10(
        train=False, root="data", transform=img_transform
    )
    train_dataloader = torch.utils.data.DataLoader(
        train_dataset,
        batch_size=256,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
    )
    val_dataloader = torch.utils.data.DataLoader(
        val_dataset,
        batch_size=256,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
    )
    criterion = nn.CrossEntropyLoss()
    NUM_EPOCHS = 10
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.003)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=NUM_EPOCHS, eta_min=1e-05
    )
    # result = TrainResult(train_losses=[], train_accs=[], val_losses=[], val_accs=[])
    for epoch in range(NUM_EPOCHS):
        loss_meter = AverageMeter()
        acc_meter = AverageMeter()
        for img, labels in tqdm.tqdm(
            train_dataloader, desc="Training at " + str(epoch)
        ):
            img, labels = (img.to(device), labels.to(device))
            optimizer.zero_grad()
            outputs, _ = model(img)
            loss = criterion(outputs, labels)
            loss_meter.update(loss.item(), len(img))
            acc = (outputs.argmax(-1) == labels).float().mean().item()
            acc_meter.update(acc, len(img))
            loss.backward()
            optimizer.step()
        scheduler.step()
        result.train_losses.append(loss_meter.calculate())
        result.train_accs.append(acc_meter.calculate())
        print(
            f"Train Epoch: {epoch}, Loss: {loss_meter.calculate()}, Acc: {acc_meter.calculate()}"
        )
        val_loss, val_acc = evaluate_cifar_model(model, criterion, val_dataloader)
        result.val_losses.append(val_loss)
        result.val_accs.append(val_acc)
        print(f"Val Epoch: {epoch}, Loss: {val_loss}, Acc: {val_acc}")
        best_models = "Vision_transformer_Best_" + str(epoch + 1) + ".pt"
        best_save_path = os.path.join(checkpoint_dir, best_models)
        model_path = "Vision_transformer_" + str(epoch + 1) + ".pt"
        save_path = os.path.join(checkpoint_dir, model_path)

        # todo add an early stopping criteria and restore best weights

        if epoch > 0 and result.val_losses[epoch] > result.val_losses[epoch - 1]:
            torch.save(model.state_dict(), best_save_path)
            val_loss, val_acc = evaluate_cifar_model(model, criterion, val_dataloader)
            print(f"Val Epoch: {epoch + 1}, Loss: {val_loss}, Acc: {val_acc}")
            print("Finished Training")

        else:
            torch.save(model.state_dict(), save_path)
        val_loss, val_acc = evaluate_cifar_model(model, criterion, val_dataloader)
        clean_memory_cache()
        print(f"Val Epoch: {epoch + 1}, Loss: {val_loss}, Acc: {val_acc}")
    print("Finished Training")

In [23]:
device

'mps'

In [43]:
main()

Training at 0:   0%|          | 0/196 [00:00<?, ?it/s]/Users/deven/.virtualenvs/Machine_Learning_Algorithms/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Training at 0: 100%|██████████| 196/196 [00:45<00:00,  4.30it/s]


Train Epoch: 0, Loss: 2.319605597991943, Acc: 0.0971600000011921
Val Epoch: 0, Loss: 2.3100959854125978, Acc: 0.1


/Users/deven/.virtualenvs/Machine_Learning_Algorithms/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Before Clearing, Available memory: 2207.17 MB
Val Epoch: 1, Loss: 2.3100959854125978, Acc: 0.1


Training at 1: 100%|██████████| 196/196 [00:48<00:00,  4.04it/s]


...

Training at 2: 100%|██████████| 196/196 [00:47<00:00,  4.11it/s]


Train Epoch: 2, Loss: 2.3080648083496094, Acc: 0.10208000000476837
Val Epoch: 2, Loss: 2.307296810913086, Acc: 0.1
Before Clearing, Available memory: 1168.27 MB
Val Epoch: 3, Loss: 2.307296810913086, Acc: 0.1


Training at 3: 100%|██████████| 196/196 [00:47<00:00,  4.12it/s]


Train Epoch: 3, Loss: 2.307045121917725, Acc: 0.10061999999046325
Val Epoch: 3, Loss: 2.3089694084167482, Acc: 0.1


KeyboardInterrupt: 

## Testing

In [33]:
num_tokens = 100
batch_size = 10
dim = 64
num_layers = 4
num_heads = 2
dummy_model = Transformer(
    dim=dim, attn_dim=32, mlp_dim=dim, num_heads=num_heads, num_layers=num_layers
).to(device)
inp = torch.randn(batch_size, num_tokens, dim).to(device)
dummy_model

Transformer(
  (layers): ModuleList(
    (0-3): 4 x AttentionResidual(
      (attn): MultiHeadedAttention(
        (qkv_projection): Linear(in_features=64, out_features=192, bias=False)
        (W0): Linear(in_features=64, out_features=64, bias=True)
      )
      (ffn): Sequential(
        (0): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
        (1): Linear(in_features=64, out_features=64, bias=True)
        (2): GELU(approximate='none')
        (3): Linear(in_features=64, out_features=64, bias=True)
      )
    )
  )
)

In [34]:
# This should print a large number (e.g., 50+).
# If it prints a small number (like 6), your Transformer layers are disconnected.
print("Total registered parameter tensors:", len(list(dummy_model.parameters())))

Total registered parameter tensors: 36


In [ ]:
# This should print a large number (e.g., 50+).
# If it prints a small number (like 6), your Transformer layers are disconnected.
print("Total registered parameter tensors:", len(list(dummy_model.parameters())))

In [35]:
dummy_model(inp)

(tensor([[[-0.6479,  1.6454,  0.8780,  ...,  0.4604, -0.2581, -2.0611],
          [ 0.6369, -0.1756,  0.1996,  ...,  0.4671,  0.4317,  0.7035],
          [-0.5313,  1.5475, -0.9728,  ...,  1.5819, -0.0643, -0.3232],
          ...,
          [-0.5979, -1.9449, -0.6834,  ..., -0.3069, -0.2602,  0.3442],
          [ 1.2375,  0.7228,  0.7221,  ...,  0.1359, -0.7810, -1.0680],
          [ 1.5384,  0.5648,  0.5669,  ...,  1.5544, -2.2397, -1.3261]],
 
         [[ 0.8371,  1.5876, -2.4725,  ..., -0.0059, -1.9571,  1.0069],
          [ 0.3596,  0.9923, -0.3046,  ...,  0.8323,  0.3619,  2.3932],
          [ 0.6751,  0.5012, -0.5788,  ...,  0.4745,  0.3530, -0.6682],
          ...,
          [ 0.9923, -1.5202, -0.8114,  ...,  0.4278,  0.4084, -1.4275],
          [-0.5611,  0.2909,  0.3196,  ...,  1.8573, -0.8421,  1.1546],
          [ 0.5476,  0.3074, -0.0436,  ...,  0.5956, -0.4679,  0.3378]],
 
         [[-1.8204, -0.8329,  0.3052,  ...,  2.0438, -0.5137, -0.8785],
          [-0.3736, -0.7255,

In [36]:
num_tokens = 100
batch_size = 10
dim = 64
num_layers = 4
num_heads = 2
dummy_model = Transformer(
    dim=dim, attn_dim=32, mlp_dim=dim, num_heads=num_heads, num_layers=num_layers
).to(device)
inp = torch.randn(batch_size, num_tokens, dim).to(device)

import torch
from torch.profiler import (
    profile,
    record_function,
    ProfilerActivity,
    tensorboard_trace_handler,
)

# --- RUN EXECUTION PROFILE ---
# The modern way: Let the profiler handle TensorBoard logging directly!
with profile(
    activities=[ProfilerActivity.CPU],
    record_shapes=True,
    with_stack=True,
    profile_memory=False,
    # This single line replaces all the Pandas and SummaryWriter logic
    on_trace_ready=tensorboard_trace_handler("./runs/transformer_profiler_metrics"),
) as prof:

    with record_function("transformer_forward"):
        if torch.backends.mps.is_available():
            torch.mps.synchronize()

        # Assuming dummy_model and inp are defined earlier in your script
        output = dummy_model(inp)

        if torch.backends.mps.is_available():
            torch.mps.synchronize()

print("Trace successfully logged to TensorBoard!")
print("To inspect the actual profiler UI, run: tensorboard --logdir=./runs")
print("Once open, navigate to the 'PYTORCH_PROFILER' tab at the top.")
# inp = torch.randn(batch_size, num_tokens, dim).to(device)
# print(f"Input: {inp.shape=}")
# # test case 1 regular forward pass
# print("Test Case 1")
# with torch.no_grad():
#     output, alpha = dummy_model(inp, attn_mask=None)
#     # print(f"\n\n\n{output.shape=},{alpha.shape=}")
#
#     # if len(alpha)==0:
#     #     alpha=None
#     assert alpha is None
#     assert output.shape == (
#         batch_size,
#         num_tokens,
#         dim,
#     ), f"wrong output shape {output.shape}"
#
# # test case 2 collect attentions

USDT:2026-09-20 21:37:24 5367:278071 SyncActivityProfilerHandler.cpp:39] profiler_start
USDT:2026-09-20 21:37:24 5367:278071 SyncActivityProfilerHandler.cpp:46] profiler_stop


Trace successfully logged to TensorBoard!
To inspect the actual profiler UI, run: tensorboard --logdir=./runs
Once open, navigate to the 'PYTORCH_PROFILER' tab at the top.


In [52]:
import torch
import torch.nn as nn
from torch.profiler import (
    profile,
    record_function,
    ProfilerActivity,
    tensorboard_trace_handler,
)

# 1. Define device configuration for Apple Silicon
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

# 2. Transformer hyperparameters and model initialization
num_tokens = 100
batch_size = 10
dim = 64
num_layers = 4
num_heads = 2
attn_dim = 32
# Assuming Transformer is defined in your environment
dummy_model = Transformer(
    dim=dim, attn_dim=32, mlp_dim=dim, num_heads=num_heads, num_layers=num_layers
).to(device)

inp = torch.randn(batch_size, num_tokens, dim).to(device)

# 3. WARMUP PASS (Crucial for MPS shader compilation)
print("Warming up MPS backend...")
with torch.no_grad():
    _ = dummy_model(
        inp,
    )
if device.type == "mps":
    torch.mps.synchronize()

# --- RUN EXECUTION PROFILE ---
print("Starting execution profile...")
with profile(
    activities=[ProfilerActivity.CPU],
    record_shapes=True,
    with_stack=True,
    profile_memory=False,
    on_trace_ready=tensorboard_trace_handler("./runs/transformer_profiler_metrics"),
) as prof:

    with record_function("transformer_forward"):
        if device.type == "mps":
            torch.mps.synchronize()

        with torch.no_grad():
            output = dummy_model(inp)

        if device.type == "mps":
            torch.mps.synchronize()

print("Trace successfully logged to TensorBoard!")
print("To inspect, run: python -m tensorboard.main --logdir=./runs --port=6007")

Using device: mps
Warming up MPS backend...
Starting execution profile...
Trace successfully logged to TensorBoard!
To inspect, run: python -m tensorboard.main --logdir=./runs --port=6007


USDT:2026-09-21 03:07:24 18093:605093 SyncActivityProfilerHandler.cpp:39] profiler_start
USDT:2026-09-21 03:07:24 18093:605093 SyncActivityProfilerHandler.cpp:46] profiler_stop


### Profiling

In [45]:
# Load the TensorBoard extension
%load_ext tensorboard

# Launch it directly inside your notebook cell
%tensorboard --logdir=./runs

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


## LLM

In [5]:
file = "shakespeare.txt"
with open(file, "r") as f:
    dialogues = f.read()

In [6]:
all_dialogues = dialogues.split("\n\n")

In [7]:
for line in all_dialogues[:10]:
    print(line)

First Citizen:
Before we proceed any further, hear me speak.
All:
Speak, speak.
First Citizen:
You are all resolved rather to die than to famish?
All:
Resolved. resolved.
First Citizen:
First, you know Caius Marcius is chief enemy to the people.
All:
We know't, we know't.
First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?
All:
No more talking on't; let it be done: away, away!
Second Citizen:
One word, good citizens.
First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.


In [8]:
import nltk

nltk.download("punkt_tab")

[nltk_data] Downloading package punkt_tab to /Users/deven/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [9]:
def tokenize(s):
    return nltk.word_tokenize(s)

## Dialogue LLM

In [10]:
def tokenize(s):
    return nltk.word_tokenize(s)


class MyTokenizer:
    def __init__(self, raw_text: str):
        # raw_text     contains the text from which we will build our vocabulary

        self.start = "<START>"  # token that starts every example
        self.pad = "<PAD>"  # token used to pad examples to the same length
        self.unk = "<UNK>"  # token used if encountering a word not in our vocabulary

        vocab = np.unique(tokenize(raw_text))
        vocab = np.concatenate([np.array([self.start, self.pad, self.unk]), vocab])

        self.vocab = vocab  # array of tokens in order
        self.tok_to_id = {w: i for i, w in enumerate(vocab)}  # mapping of token to ID
        self.id_to_token = {i: w for i, w in enumerate(vocab)}
        self.vocab_size = len(self.vocab)  # size of vocabulary

    def __len__(self):
        return self.vocab_size

    def encode(self, s: str) -> torch.Tensor:
        # s           input string
        #
        # Output
        # id_tensor   a tensor of token ids, starting with the start token.t

        id_tensor = torch.from_numpy(
            np.array(
                [self.tok_to_id[self.start]]
                + [self.tok_to_id[w] for w in tokenize(s) if w in self.tok_to_id],
                dtype=np.int32,
            )
        )

        # TODO: tokenize the input using word_tokenize. Return a tensor  of the token ids, starting with the token id for the start token.
        # ============ ANSWER START ===========
        # encoded_string = tokenize(s)
        # token_ids =
        # token_ids.append(self.tok_to_id[self.start])
        # token_ids.extend(
        #     [self.tok_to_id[w] for w in encoded_string if w in self.tok_to_id.keys()]
        # )
        # id_tensor = np.array(token_ids)

        # id_tensor = torch.from_numpy(id_tensor)
        # ============ ANSWER END =============

        return id_tensor

    def decode(self, toks: torch.Tensor) -> str:
        # toks         a list of token ids
        #
        # Output
        # decoded_str  the token ids decoded back into a string (join with a space)

        # TODO: convert the token ids back to the actual corresponding words.
        # Join the tokens with a space and return the full string
        # ============ ANSWER START ===========
        return " ".join(
            [
                self.id_to_token[int(token)]
                for token in toks
                if token in self.tok_to_id.values()
            ]
        ).rstrip()

        # ============ ANSWER END =============

        # return decoded_str

    def pad_examples(self, tok_list: List[torch.Tensor]) -> torch.Tensor:
        # Pads the tensors to the right with the pad token so that they are the same length.
        #
        # tok_list       a list of tensors containing token ids (maybe of different lengths)
        #
        # Output
        # padded_tokens  shape: (len(tok_list), max length within tok_list)
        return torch.nn.utils.rnn.pad_sequence(
            tok_list, batch_first=True, padding_value=self.tok_to_id[self.pad]
        )


tok = MyTokenizer(dialogues)

In [11]:
len(tok)

14058

In [12]:
# tokenizer test cases
input_string = "KING RICHARD III:\nSay that I did all this for love of her. bluye"
enc = tok.encode(input_string)
print(enc)

# for x in enc:
#     print(x, type(x), x in tok.tok_to_id.values(), x in tok.id_to_token.values())
dec = tok.decode(enc)
print(dec)
# print("<START> KING RICHARD III : Say that I did all this for love of her .")
assert dec == "<START> KING RICHARD III : Say that I did all this for love of her ."

tensor([    0,  1396,  1986,  1284,    18,  2148, 12580,  1279,  5523,  3135,
        12633,  6669,  8559,  9443,  7480,    16], dtype=torch.int32)
<START> KING RICHARD III : Say that I did all this for love of her .


In [50]:
# num_tokens = 100
# batch_size = 10
# dim = 64
# num_layers = 4
# num_heads = 2
#
# # Assuming Transformer is defined in your environment
# dummy_model = Transformer(
#     dim=dim, attn_dim=32, mlp_dim=dim, num_heads=num_heads, num_layers=num_layers
# ).to(device)

In [13]:
class DialogueDataset:
    def __init__(self, tokenizer: MyTokenizer, lines: List[str], max_N: int):
        # tokenizer    an instance of MyTokenizer
        # lines        a list of strings. each element in an example in the dataset
        # max_N        the maximum number of tokens allowed per example. More than this will be truncated
        self.lines = lines
        self.tokenizer = tokenizer
        self.max_N = max_N

    def __len__(self) -> int:
        return len(self.lines)

    # def __iter__(self):
    #     for line in self.lines:
    #         yield self.tokenizer.encode(line)[: self.max_N]

    def __getitem__(self, idx: int) -> torch.Tensor:
        # returns the example at int encoded by the tokenizer
        # truncates the example if it is more than max_N tokens
        return self.tokenizer.encode(self.lines[idx])[: self.max_N]

    # def __getitems__(self,indices:int):
    #     return [self.__getitem__(idx) for idx in indices]

In [14]:
ds = DialogueDataset(tok, all_dialogues, max_N=200)

In [15]:
def collate_fn(examples: List[torch.Tensor]):
    """
    # examples        a batch of tensors containing token ids (maybe of different lengths)
    # Outputs a dictionary containing
    #   input_ids     a single tensor with all of the examples padded (from the right) to the max
    #                 length within the batch. shape:(B, max length within examples)
    #   input_mask    a tensor indicating which tokens are padding and should be ignored. 0 if padding
    #                 and 1 if not. shape: (B, max length within examples)
    """
    new_input_ids = tok.pad_examples(examples)
    attn_mask = torch.ones(new_input_ids.shape)  # 1s should not be ignored

    # causal attention mask
    attn_mask[new_input_ids == tok.tok_to_id[tok.pad]] = (
        0  # should be ignored if it is a padded
    )
    return {"input_ids": new_input_ids, "input_mask": attn_mask}

In [16]:
ds[1]

tensor([    0,   118,    18,  2324,    14, 11846,    16], dtype=torch.int32)

In [17]:
tokens = [ds[1], ds[2]]

In [18]:
type(tokens)

list

In [19]:
collate_fn(tokens)

{'input_ids': tensor([[    0,   118,    18,  2324,    14, 11846,    16,     1,     1,     1,
              1,     1,     1,     1,     1],
         [    0,   950,   505,    18,  2865,  3322,  3135, 10820, 10561, 12760,
           5525, 12571, 12760,  6328,    20]], dtype=torch.int32),
 'input_mask': tensor([[1., 1., 1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
         [1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]])}

In [20]:
tokens[0]

tensor([    0,   118,    18,  2324,    14, 11846,    16], dtype=torch.int32)

In [21]:
tokens[1]

tensor([    0,   950,   505,    18,  2865,  3322,  3135, 10820, 10561, 12760,
         5525, 12571, 12760,  6328,    20], dtype=torch.int32)

In [22]:
from torch.utils.data import DataLoader, Subset
import torch.nn.functional as F

In [23]:
tok

In [24]:
train_dl = DataLoader(ds, batch_size=16, num_workers=0, collate_fn=collate_fn)

In [25]:
BATCH_SIZE = 16
BUFFER_SIZE = 4

## DialogueGPT Model

In [78]:
from typing import Optional, Tuple, Any, List
import torch
import torch.nn as nn
from dataclasses import dataclass

import numpy as np
import torch
import torchvision
from torch import nn, optim
from torchvision import transforms
import sklearn
from sklearn.metrics import confusion_matrix
import tqdm
import copy
from torch.utils.data import DataLoader, Subset
import torch.nn.functional as F

torch.autograd.set_detect_anomaly(True)
device = (
    torch.accelerator.current_accelerator().type
    if torch.accelerator.is_available()
    else "cpu"
)
import os

num_workers = min(2, os.cpu_count())
import logging

# Configure logging to write directly to a file
logging.basicConfig(
    filename="vit.log",
    filemode="a",  # 'a' appends to the file, 'w' overwrites it on every run
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)
import gc
import torch


def clean_memory_cache():

    if torch.mps.is_available():
        print(
            f"Before Clearing, Available memory: {torch.mps.driver_allocated_memory() / 1024 / 1024:.2f} MB"
        )
        gc.collect()
        torch.mps.empty_cache()
    elif torch.cuda.is_available():
        print(
            f"Before Clearing, Available memory: {torch.cuda.memory_allocated() / 1024 / 1024:.2f} MB"
        )
        gc.collect()
        torch.cuda.empty_cache()
    else:
        gc.collect()
        return


@dataclass(init=True)
class TrainResult:
    """
    A collection containing everything we need to know about the training results
    """

    train_losses: List[float]
    train_accs: List[float]
    val_accs: List[float]
    val_losses: List[float]


result = TrainResult(train_losses=[], train_accs=[], val_accs=[], val_losses=[])


class AverageMeter:
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0.0
        self.avg = 0.0
        self.sum = 0.0
        self.count = 0.0

    def update(self, val: float, n: int = 1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count if self.count > 0 else 0.0

    def calculate(self) -> float:
        return self.avg


def print_variance(name: str, data: torch.Tensor):
    # Compute variance across features/neurons and average across the batch
    neuron_variance = torch.mean(torch.var(data.detach().float(), dim=-1))
    print(f"{name}: Variance = {neuron_variance.item():.6f}")


import torch
import torch.nn as nn


class OptimizedAttentionLayer(nn.Module):
    def __init__(self, dim, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads

        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)
        self.out_proj = nn.Linear(dim, dim)

    def forward(self, x):
        batch_size, seq_len, dim = x.shape

        # Project and reshape for multi-head attention
        q = (
            self.q_proj(x)
            .view(batch_size, seq_len, self.num_heads, self.head_dim)
            .transpose(1, 2)
        )
        k = (
            self.k_proj(x)
            .view(batch_size, seq_len, self.num_heads, self.head_dim)
            .transpose(1, 2)
        )
        v = (
            self.v_proj(x)
            .view(batch_size, seq_len, self.num_heads, self.head_dim)
            .transpose(1, 2)
        )

        # --- THE MODERN OPTIMIZATION ---
        # Instead of manual matmul + scale + masked_fill + softmax + bmm,
        # use PyTorch's native SDPA scaled dot product attention. It automatically fuses these kernels
        # and triggers FlashAttention under the hood if hardware allows!
        out = F.scaled_dot_product_attention(q, k, v, is_causal=False)

        # Reshape back
        out = out.transpose(1, 2).contiguous().view(batch_size, seq_len, dim)
        return self.out_proj(out)


class MultiHeadedAttention(nn.Module):
    def __init__(self, dim: int, n_hidden: int, num_heads: int):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.n_hidden = n_hidden
        self.scale = n_hidden**-0.5

        self.qkv_projection = nn.Linear(dim, num_heads * n_hidden * 3, bias=False)
        self.W0 = nn.Linear(num_heads * n_hidden, dim)

    def forward(
        self,
        x: torch.Tensor,
        attn_mask: Optional[torch.Tensor] = None,
        layer_past: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
        use_cache: bool = False,
    ) -> Tuple[torch.Tensor, Optional[Tuple[torch.Tensor, torch.Tensor]]]:
        B, T, _ = x.shape

        qkv = (
            self.qkv_projection(x)
            .reshape(B, T, self.num_heads, 3 * self.n_hidden)
            .transpose(1, 2)
        )
        q, k, v = qkv.chunk(3, dim=-1)

        if layer_past is not None:
            past_k, past_v = layer_past
            k = torch.cat([past_k, k], dim=-2)
            v = torch.cat([past_v, v], dim=-2)

        present = (k, v) if use_cache else None

        # --- THE BROADCASTING FIX ---
        if attn_mask is not None:
            # If mask is (B, T, T), expand to (B, 1, T, T)
            if attn_mask.dim() == 3:
                attn_mask = attn_mask.unsqueeze(1)
            # If mask is (T, T), expand to (1, 1, T, T)
            elif attn_mask.dim() == 2:
                attn_mask = attn_mask.unsqueeze(0).unsqueeze(0)

        # When using an explicit attn_mask, is_causal MUST be False
        is_causal = (attn_mask is None) and (T > 1) and (layer_past is None)

        context = F.scaled_dot_product_attention(
            q, k, v, attn_mask=attn_mask, is_causal=is_causal
        )

        context = (
            context.transpose(1, 2)
            .contiguous()
            .view(B, T, self.num_heads * self.n_hidden)
        )
        output = self.W0(context)
        return output, present


class AttentionResidual(nn.Module):
    def __init__(self, dim: int, attn_dim: int, mlp_dim: int, num_heads: int):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)

        self.attn = MultiHeadedAttention(dim, attn_dim, num_heads)
        # self.attn = OptimizedAttentionLayer(dim,attn_dim, num_heads)
        # LayerNorm applied inside the FFN sequence only
        self.norm2 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(
            # nn.LayerNorm(dim),
            nn.Linear(dim, mlp_dim),
            nn.GELU(),
            nn.Linear(mlp_dim, dim),
        )

    def forward(
        self,
        x: torch.Tensor,
        attn_mask: Optional[torch.Tensor] = None,
        layer_past: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
        use_cache: bool = False,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        # Attention block with residual connection (no norm)
        attn_out, alphas = self.attn(
            x=self.norm1(x),
            attn_mask=attn_mask,
            layer_past=layer_past,
            use_cache=use_cache,
        )
        x = x + attn_out

        # FFN block with residual connection (norm is first layer inside self.ffn)
        x = x + self.ffn(self.norm2(x))
        return x, alphas


class Transformer(nn.Module):
    def __init__(
        self, dim: int, attn_dim: int, mlp_dim: int, num_heads: int, num_layers: int
    ):
        super().__init__()
        self.layers = nn.ModuleList(
            [
                AttentionResidual(dim, attn_dim, mlp_dim, num_heads)
                for _ in range(num_layers)
            ]
        )

    def forward(
        self,
        x: torch.Tensor,
        attn_mask: Optional[torch.Tensor] = None,
        return_attn: bool = False,
        layer_past: Optional[List[Tuple[torch.Tensor, torch.Tensor]]] = None,
        use_cache: bool = False,
    ) -> Tuple[
        torch.Tensor,
        Optional[torch.Tensor],
        Optional[List[Tuple[torch.Tensor, torch.Tensor]]],
    ]:
        presents = [] if use_cache else None

        for i, layer in enumerate(self.layers):
            # Extract layer-specific cache safely
            past_kv = layer_past[i] if layer_past is not None else None

            x, present_kv = layer(
                x,
                attn_mask=attn_mask,
                layer_past=past_kv,
                use_cache=use_cache,
            )
            if use_cache:
                presents.append(present_kv)

        return x, None, presents


class DialogueGPT(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        max_N: int,
        dim: int,
        attn_dim: int,
        mlp_dim: int,
        num_heads: int,
        num_layers: int,
    ):
        super().__init__()
        self.token_embeddings = nn.Embedding(vocab_size, dim)
        self.pos_embeddings = nn.Embedding(max_N, dim)
        self.transformer = Transformer(
            dim=dim,
            attn_dim=attn_dim,
            mlp_dim=mlp_dim,
            num_heads=num_heads,
            num_layers=num_layers,
        )
        self.head = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, vocab_size))

    def forward(
        self,
        input_ids: torch.Tensor,
        return_attn: bool = False,
        layer_past: Optional[List[Tuple[torch.Tensor, torch.Tensor]]] = None,
        use_cache: bool = False,
    ):
        B, T = input_ids.shape

        # Offset positions by the number of cached tokens
        past_length = layer_past[0][0].shape[-2] if layer_past is not None else 0
        pos_ids = torch.arange(
            past_length, past_length + T, dtype=torch.long, device=input_ids.device
        ).unsqueeze(0)

        embs = self.token_embeddings(input_ids) + self.pos_embeddings(pos_ids)

        # Causal mask is only needed for sequences longer than 1 when NOT using cache
        if layer_past is None and T > 1:
            causal_attn_mask = (
                torch.tril(torch.ones(T, T, device=input_ids.device))
                .unsqueeze(0)
                .repeat(B, 1, 1)
            ) == 1
        else:
            causal_attn_mask = None

        x, alphas, presents = self.transformer(
            embs,
            attn_mask=causal_attn_mask,
            return_attn=return_attn,
            layer_past=layer_past,
            use_cache=use_cache,
        )
        out = self.head(x)

        # Return presents ONLY when caching is explicitly requested
        if use_cache:
            return out, alphas, presents
        return out, alphas

    def key_value_cached_generation(self, input_ids: torch.Tensor, num_tokens: int):
        with torch.no_grad():
            # 1. Pre-fill: process prompt, obtain initial cache
            out, _, cache = self.forward(input_ids, use_cache=True)
            new_token = torch.argmax(out[:, [-1]], dim=-1)
            input_ids = torch.cat([input_ids, new_token], dim=1)

            # 2. Decode: pass only the single newest token and update cache
            for _ in range(num_tokens - 1):
                out, _, cache = self.forward(
                    new_token, layer_past=cache, use_cache=True
                )
                new_token = torch.argmax(out[:, [-1]], dim=-1)
                input_ids = torch.cat([input_ids, new_token], dim=1)

        return input_ids


class DialogueLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.criterion = nn.CrossEntropyLoss(reduction="none")

    def forward(
        self, logits: torch.Tensor, input_ids: torch.Tensor, inp_mask: torch.Tensor
    ):
        """
        # logits      the logits produced by DialogueGPT. shape: (B x T x V)
        # input_ids   the token ids. shape: (B x T)
        # inp_mask    a 0/1 mask of which tokens are padding tokens and should be ignored. shape: (B x T)

        TODO: Implement the language model loss. For logits[i], we want to supervise the i+1 token_id with the cross entropy loss. We thus will not supervise the start token (input_ids[0]) or use the last logit vector (logits[-1]). Return the average of the losses for each token in the batch, making sure to ignore tokens corresponding to the padding (use inp_mask).
        """
        loss = 0

        # start_token = input_ids[0]
        relevant_logits = logits[:, :-1, :]
        # print(f"{logits.shape=},{relevant_logits.shape=}")
        relevant_tokens = input_ids[:, 1:]
        shift_mask = inp_mask[:, 1:]
        relevant_logits = relevant_logits.permute(0, 2, 1)
        # print(f"{relevant_logits.shape=}")

        loss = self.criterion(relevant_logits, relevant_tokens)
        shift_mask = shift_mask.to(dtype=loss.dtype, device=loss.device)
        loss *= shift_mask

        return torch.sum(loss) / torch.clamp(torch.sum(shift_mask), min=1e-9)


import torch.optim as optim

model = DialogueGPT(
    vocab_size=tok.vocab_size,
    max_N=200,
    dim=128,
    attn_dim=64,
    mlp_dim=128,
    num_heads=3,
    num_layers=6,
).to(device)
criterion = DialogueLoss()

NUM_EPOCHS = 80

optimizer = optim.AdamW(
    model.parameters(), lr=0.0001, weight_decay=0
)  # implement in homework
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

# Ensure optimizer starts completely clean before the epoch loop begins
optimizer.zero_grad()

for epoch in range(NUM_EPOCHS):
    loss_meter = AverageMeter()

    # Wrap your loader securely
    for step, inp_dict in tqdm.tqdm(
        enumerate(train_dl), desc=f"Training at {epoch}", total=len(train_dl)
    ):
        inp_ids, inp_mask = inp_dict["input_ids"], inp_dict["input_mask"]

        inp_ids = inp_ids.to(device).long()
        inp_mask = inp_mask.to(device)

        # 1. Forward Pass
        outputs, _ = model(input_ids=inp_ids)

        # 2. FIX: Scale the loss down by BUFFER_SIZE to normalize gradients
        loss = criterion(outputs, inp_ids, inp_mask)
        scaled_loss = loss / BUFFER_SIZE

        # 3. Backward Pass (Accumulates gradients safely)
        scaled_loss.backward()

        # Track the true unscaled loss in your meter
        loss_meter.update(loss.item(), len(inp_dict["input_ids"]))

        # 4. FIX: Step the optimizer on buffer limit OR at the final step of the dataset
        if (step + 1) % BUFFER_SIZE == 0 or (step + 1) == len(train_dl):
            optimizer.step()
            optimizer.zero_grad()  # Resets the buffer for the next accumulation block

            # OPTIONAL: If your scheduler decays per-step instead of per-epoch,
            # place `scheduler.step()` right here.

    # 5. Step scheduler at the epoch level (if using an epoch-based scheduler)
    scheduler.step()

    # 6. Safe Text Generation Example (Switched to eval/inference mode to protect memory buffers)
    model.eval()
    with torch.inference_mode():
        # Ensure your start prompt token matches your vocabulary bounds
        inp = tok.encode("").unsqueeze(0).to(device)
        generated_output = model.key_value_cached_generation(inp, 10)
        print(f"\n[Generated Sample]: {tok.decode(generated_output[0][1:].cpu())}")

    model.train()  # Switch back to training mode for the next epoch loop
    clean_memory_cache()

    print(
        f"Train Epoch: {epoch}, Loss: {loss_meter.calculate():0.4f}, LR: {scheduler.get_last_lr()[0]}"
    )

In [80]:
dummy_model = Transformer(
    dim=dim, attn_dim=32, mlp_dim=dim, num_heads=num_heads, num_layers=num_layers
).to(device)

In [81]:
dummy_model(inp, layer_past=None, use_cache=False)

(tensor([[[-0.3571, -0.4444, -0.3373,  ..., -0.0735,  0.1670,  0.4280],
          [-0.9354, -1.4477, -0.5804,  ..., -0.3998,  0.5712,  0.1154],
          [ 0.3778,  1.3756,  0.5040,  ..., -0.5698, -1.2264,  1.0089],
          ...,
          [-1.1347, -1.8321, -0.0663,  ..., -0.1906, -1.9118, -0.2106],
          [-0.5117,  0.3731, -1.5395,  ...,  0.8055, -1.0320, -0.5782],
          [ 0.4691, -2.2225,  0.1292,  ...,  0.0685,  1.8795, -0.9833]],
 
         [[ 2.1419, -1.2117, -1.4154,  ...,  1.2971,  1.8064,  1.1184],
          [ 1.1750, -0.3736,  0.5764,  ..., -0.6991, -0.5160,  1.6321],
          [-0.5197, -0.0090,  2.2153,  ..., -1.2120,  0.6949, -0.9274],
          ...,
          [-1.2912,  0.4102, -0.8045,  ..., -1.8282,  0.6338, -0.2343],
          [-0.4197,  2.7169, -0.4722,  ...,  2.0855,  0.5302,  1.4296],
          [-0.7892,  0.3945,  1.2133,  ..., -0.4774,  1.6072, -1.0619]],
 
         [[-1.1401,  2.1803, -0.4098,  ...,  1.1230, -0.9461,  1.4757],
          [-2.5165,  1.1191,

In [82]:
# class DialogueGPT(nn.Module):
#     def __init__(
#         self,
#         vocab_size: int,
#         max_N: int,
#         dim: int,
#         attn_dim: int,
#         mlp_dim: int,
#         num_heads: int,
#         num_layers: int,
#     ):
#         # vocab_size       size of the vocabulary
#         # max_N            maximum number of tokens allowed to appear in 1 example
#         # dim              embedding dimension
#         # attn_dim         the hidden dimension of the attention layer
#         # mlp_dim          the hidden layer dimension of the FFN
#         # num_heads        the number of heads in the attention layer
#         # num_layers       the number of attention layers.
#
#         super().__init__()
#         """
#         • Given the token ids, retrieve the corresponding token embeddings. Add to this a learned positional embedding.
#         • Generate a causal attention mask. Remember that for GPT, every token only depends on itself and the tokens before it
#         • Pass the embeddings and the attention mask to the transformer and the language model head. Output logits of size (T ×V) where T is the number of tokens and V is the vocabulary size. This step is implemented for you
#         """
#         # TODO: set up the token embedding and positional embeddings
#         #       Hint, use nn.Embedding
#         # Already Padded to keep sequence length same
#         # Next use Graph Neural Network
#         # token_embeddings
#
#         self.token_embeddings = nn.Embedding(
#             num_embeddings=vocab_size, embedding_dim=dim
#         )
#
#         self.pos_embeddings = nn.Embedding(num_embeddings=max_N, embedding_dim=dim)
#
#         self.transformer = Transformer(
#             dim=dim,
#             attn_dim=attn_dim,
#             mlp_dim=mlp_dim,
#             num_heads=num_heads,
#             num_layers=num_layers,
#         )
#         # Projection Head
#         self.head = nn.Sequential(nn.LayerNorm(dim),
#                                   nn.Linear(dim, vocab_size))
#
#     # def forward(
#     #     self, input_ids: torch.Tensor, return_attn=False
#     # ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
#     #     # input_ids     a batch of input ids (right padded). shape: (B x T)
#     #     # return_attn   whether to return the attention weights
#     #     #
#     #     # Output
#     #     # out           the logit vector (B x T x V)
#     #     # alphas        the attention weights if return_attn is True. Otherwise None shape: (B, num_layers, num_heads, T, T)
#     #     """
#     #
#     #     Args:
#     #         input_ids: Batch of input ids right padded shape: (BxT)
#     #         return_attn: whether to return the attention weights
#     #     • Generate a causal attention mask. Remember that for Generative  Pretrained Transformer, every token only depends on itself and the tokens before it
#     #     • Pass the embeddings and the attention mask to the transformer and the language model head. Output logits of size (T ×V) where T is the number of tokens and V is the vocabulary size. This step is implemented for you
#     #     Returns:
#     #
#     #     """
#     #     B, T = input_ids.shape
#     #     pos_ids = torch.arange(0, T, dtype=torch.long, device=device).unsqueeze(
#     #         0
#     #     )  # Shape: (1, T)
#     #     embs = self.token_embeddings(input_ids) + self.pos_embeddings(pos_ids)
#     #     # TODO: retrieve the token embeddings for the input_ids.
#     #     #       Add to the token embeddings the positional embeddings.
#     #     #       Store the combined embedding in embs
#     #
#     #     # TODO: Create the causal attention mask, which should be of size (B, T, T)
#     #     #       Remember that the causal attention mask is lower triangular (all tokens only
#     #     #       depend on themselves and the tokens before them).   Store the mask in causal_attn_mask
#     #     # Hint: check out torch.tril creates a causal mask where each token can only attend to previous tokens and itself
#     #     causal_attn_mask = (
#     #         torch.tril(torch.ones(T, T)).unsqueeze(0).repeat(B, 1, 1)
#     #     ).to(
#     #         device
#     #     )  # Shape: (B, T, T)
#     #     # ============ ANSWER START ============
#     #
#     #     # ============ ANSWER END ==============
#     #
#     #     x, alphas = self.transformer(
#     #         embs, attn_mask=causal_attn_mask, return_attn=return_attn
#     #     )
#     #     out = self.head(x)
#     #     return out, alphas
#     def forward(
#             self,
#             input_ids: torch.Tensor,
#             return_attn=False,
#             layer_past=None,
#             use_cache=False
#     ) -> Tuple[torch.Tensor, Optional[torch.Tensor], Optional[Any]]:
#
#         B, T = input_ids.shape
#
#         # 1. Calculate positional offset based on the cache size
#         past_length = 0
#         if layer_past is not None:
#             # layer_past is a list of tuples: [(past_k, past_v), ...]
#             # past_k shape: (B, num_heads, past_seq_len, head_dim)
#             past_length = layer_past[0][0].shape[-2]
#
#         # Positional IDs now properly count forward from the end of the cache
#         pos_ids = torch.arange(
#             past_length, past_length + T, dtype=torch.long, device=input_ids.device
#         ).unsqueeze(0)
#
#         embs = self.token_embeddings(input_ids) + self.pos_embeddings(pos_ids)
#
#         # 2. Causal Mask Generation (SDPA prefers boolean masks)
#         causal_attn_mask = (
#                                torch.tril(torch.ones(T, T)).unsqueeze(0).repeat(B, 1, 1)
#                            ).to(input_ids.device) == 1
#
#         # 3. Route cache through the transformer (assuming Transformer class is updated)
#         x, alphas, present_cache = self.transformer(
#             embs,
#             attn_mask=causal_attn_mask,
#             return_attn=return_attn,
#             layer_past=layer_past,
#             use_cache=use_cache
#         )
#
#         out = self.head(x)
#         return out, alphas, present_cache
#
# def key_value_cached_generation(self, input_ids, num_tokens):
#     # you can assume batch size 1
#     # greedy generation with KV cache
#     with torch.no_grad():
#         # 1. PRE-FILL PHASE
#         # Ingest the entire initial prompt to build the first KV cache
#         out, _, cache = self.forward(input_ids, use_cache=True)
#
#         # Extract the prediction for the last token in the prompt
#         new_token = torch.argmax(out[:, [-1]], dim=-1)
#
#         # Append it to our running sequence
#         input_ids = torch.cat([input_ids, new_token], dim=1)
#
#         # 2. DECODE PHASE
#         # Generate the remaining (num_tokens - 1) tokens
#         for _ in range(num_tokens - 1):
#             # CRITICAL: We DO NOT pass 'input_ids' anymore.
#             # We ONLY pass the single 'new_token' and the 'cache'.
#             out, _, cache = self.forward(
#                 new_token,
#                 layer_past=cache,
#                 use_cache=True
#             )
#
#             # The output is now shape (B, 1, Vocab_Size). Grab the prediction.
#             new_token = torch.argmax(out[:, [-1]], dim=-1)
#
#             # Append the new word to the final sequence
#             input_ids = torch.cat([input_ids, new_token], dim=1)
#
#     return input_ids
#     def generate(self, input_ids, num_tokens):
#         # you can assume batch size 1
#         # greedy generation
#         with torch.no_grad():
#             for i in range(num_tokens):
#                 out, _ = self.forward(input_ids)
#                 new_token = torch.argmax(out[:, [-1]], -1)
#                 input_ids = torch.cat([input_ids, new_token], dim=1)
#         return input_ids
#
#     # def key_value_cached_generation(self, input_ids, num_tokens, cache):
#     #     pass

In [83]:
# class DialogueGPT(nn.Module):
#     def __init__(
#             self,
#             vocab_size: int,
#             max_N: int,
#             dim: int,
#             attn_dim: int,
#             mlp_dim: int,
#             num_heads: int,
#             num_layers: int,
#     ):
#         super().__init__()
#         self.token_embeddings = nn.Embedding(vocab_size, dim)
#         self.pos_embeddings = nn.Embedding(max_N, dim)
#         self.transformer = Transformer(
#             dim=dim,
#             attn_dim=attn_dim,
#             mlp_dim=mlp_dim,
#             num_heads=num_heads,
#             num_layers=num_layers,
#         )
#         self.head = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, vocab_size))
#
#     def forward(
#             self,
#             input_ids: torch.Tensor,
#             return_attn: bool = False,
#             layer_past: Optional[List[Tuple[torch.Tensor, torch.Tensor]]] = None,
#             use_cache: bool = False,
#     ):
#         B, T = input_ids.shape
#
#         # Offset positions by the number of cached tokens
#         past_length = layer_past[0][0].shape[-2] if layer_past is not None else 0
#         pos_ids = torch.arange(
#             past_length, past_length + T, dtype=torch.long, device=input_ids.device
#         ).unsqueeze(0)
#
#         embs = self.token_embeddings(input_ids) + self.pos_embeddings(pos_ids)
#
#         # Causal mask is only needed for sequences longer than 1 when NOT using cache
#         if layer_past is None and T > 1:
#             causal_attn_mask = (
#                                    torch.tril(torch.ones(T, T, device=input_ids.device))
#                                    .unsqueeze(0)
#                                    .repeat(B, 1, 1)
#                                ) == 1
#         else:
#             causal_attn_mask = None
#
#         x, alphas, presents = self.transformer(
#             embs,
#             attn_mask=causal_attn_mask,
#             return_attn=return_attn,
#             layer_past=layer_past,
#             use_cache=use_cache,
#         )
#         out = self.head(x)
#
#         # Return presents ONLY when caching is explicitly requested
#         if use_cache:
#             return out, alphas, presents
#         return out, alphas
#
#     def key_value_cached_generation(self, input_ids: torch.Tensor, num_tokens: int):
#         with torch.no_grad():
#             # 1. Pre-fill: process prompt, obtain initial cache
#             out, _, cache = self.forward(input_ids, use_cache=True)
#             new_token = torch.argmax(out[:, [-1]], dim=-1)
#             input_ids = torch.cat([input_ids, new_token], dim=1)
#
#             # 2. Decode: pass only the single newest token and update cache
#             for _ in range(num_tokens - 1):
#                 out, _, cache = self.forward(
#                     new_token, layer_past=cache, use_cache=True
#                 )
#                 new_token = torch.argmax(out[:, [-1]], dim=-1)
#                 input_ids = torch.cat([input_ids, new_token], dim=1)
#
#         return input_ids

In [84]:
# class DialogueLoss(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.criterion = nn.CrossEntropyLoss(reduction="none")
#
#     def forward(
#         self, logits: torch.Tensor, input_ids: torch.Tensor, inp_mask: torch.Tensor
#     ):
#         """
#         # logits      the logits produced by DialogueGPT. shape: (B x T x V)
#         # input_ids   the token ids. shape: (B x T)
#         # inp_mask    a 0/1 mask of which tokens are padding tokens and should be ignored. shape: (B x T)
#
#         TODO: Implement the language model loss. For logits[i], we want to supervise the i+1 token_id with the cross entropy loss. We thus will not supervise the start token (input_ids[0]) or use the last logit vector (logits[-1]). Return the average of the losses for each token in the batch, making sure to ignore tokens corresponding to the padding (use inp_mask).
#         """
#         loss = 0
#
#         # start_token = input_ids[0]
#         relevant_logits = logits[:, :-1, :]
#         # print(f"{logits.shape=},{relevant_logits.shape=}")
#         relevant_tokens = input_ids[:, 1:]
#         shift_mask = inp_mask[:, 1:]
#         relevant_logits = relevant_logits.permute(0, 2, 1)
#         # print(f"{relevant_logits.shape=}")
#
#         loss = self.criterion(relevant_logits, relevant_tokens)
#         shift_mask = shift_mask.to(dtype=loss.dtype, device=loss.device)
#         loss *= shift_mask
#
#         return torch.sum(loss) / torch.clamp(torch.sum(shift_mask), min=1e-9)

In [85]:
# def sample_next_token(input_tokens, model, tokenizer):
#     # Run model to get prediction over next output
#     outputs = model(
#         input_ids=input_tokens["input_ids"],
#         attention_mask=input_tokens["input_mask"],
#     )
#     # Find prediction
#     prob_over_tokens = F.softmax(outputs.logits, dim=-1).detach().numpy()[0, -1]
#     # TODO Draw a random token according to the probabilities
#     # next_token should be an array with an sole integer in it (as below)
#     # Use:  https://numpy.org/doc/stable/reference/random/generated/numpy.random.choice.html
#     # Replace this line
#     next_token = [5000]
#
#     # Append token to sentence
#     output_tokens = input_tokens
#     output_tokens["input_ids"] = torch.cat(
#         (output_tokens["input_ids"], torch.tensor([next_token])), dim=1
#     )
#     output_tokens["attention_mask"] = torch.cat(
#         (output_tokens["attention_mask"], torch.tensor([[1]])), dim=1
#     )
#     output_tokens["last_token_prob"] = prob_over_tokens[next_token]
#
#     return output_tokens

In [89]:
# import torch.optim as optim
#
# model = DialogueGPT(
#     vocab_size=tok.vocab_size,
#     max_N=200,
#     dim=128,
#     attn_dim=64,
#     mlp_dim=128,
#     num_heads=3,
#     num_layers=6,
# ).to(device)
# criterion = DialogueLoss()
#
# NUM_EPOCHS = 80
#
# optimizer = optim.AdamW(
#     model.parameters(), lr=0.0001, weight_decay=0
# )  # implement in homework
# scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

In [90]:
model

DialogueGPT(
  (token_embeddings): Embedding(14058, 128)
  (pos_embeddings): Embedding(200, 128)
  (transformer): Transformer(
    (layers): ModuleList(
      (0-5): 6 x AttentionResidual(
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): MultiHeadedAttention(
          (qkv_projection): Linear(in_features=128, out_features=576, bias=False)
          (W0): Linear(in_features=192, out_features=128, bias=True)
        )
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
        (ffn): Sequential(
          (0): Linear(in_features=128, out_features=128, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=128, out_features=128, bias=True)
        )
      )
    )
  )
  (head): Sequential(
    (0): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
    (1): Linear(in_features=128, out_features=14058, bias=True)
  )
)

In [ ]:
# Ensure optimizer starts completely clean before the epoch loop begins
optimizer.zero_grad()

for epoch in range(NUM_EPOCHS):
    loss_meter = AverageMeter()

    # Wrap your loader securely
    for step, inp_dict in tqdm.tqdm(
        enumerate(train_dl), desc=f"Training at {epoch}", total=len(train_dl)
    ):
        inp_ids, inp_mask = inp_dict["input_ids"], inp_dict["input_mask"]

        inp_ids = inp_ids.to(device).long()
        inp_mask = inp_mask.to(device)

        # 1. Forward Pass
        outputs, _ = model(input_ids=inp_ids)

        # 2. FIX: Scale the loss down by BUFFER_SIZE to normalize gradients
        loss = criterion(outputs, inp_ids, inp_mask)
        scaled_loss = loss / BUFFER_SIZE

        # 3. Backward Pass (Accumulates gradients safely)
        scaled_loss.backward()

        # Track the true unscaled loss in your meter
        loss_meter.update(loss.item(), len(inp_dict["input_ids"]))

        # 4. FIX: Step the optimizer on buffer limit OR at the final step of the dataset
        if (step + 1) % BUFFER_SIZE == 0 or (step + 1) == len(train_dl):
            optimizer.step()
            optimizer.zero_grad()  # Resets the buffer for the next accumulation block

            # OPTIONAL: If your scheduler decays per-step instead of per-epoch,
            # place `scheduler.step()` right here.

    # 5. Step scheduler at the epoch level (if using an epoch-based scheduler)
    scheduler.step()

    # 6. Safe Text Generation Example (Switched to eval/inference mode to protect memory buffers)
    model.eval()
    with torch.inference_mode():
        # Ensure your start prompt token matches your vocabulary bounds
        inp = tok.encode("").unsqueeze(0).to(device)
        generated_output = model.key_value_cached_generation(inp, 10)
        print(f"\n[Generated Sample]: {tok.decode(generated_output[0][1:].cpu())}")

    model.train()  # Switch back to training mode for the next epoch loop
    clean_memory_cache()

    print(
        f"Train Epoch: {epoch}, Loss: {loss_meter.calculate():0.4f}, LR: {scheduler.get_last_lr()[0]}"
    )

Training at 0: 100%|██████████| 452/452 [01:20<00:00,  5.65it/s]



[Generated Sample]: : : : , , , , , , ,
Before Clearing, Available memory: 18460.88 MB
Train Epoch: 0, Loss: 8.5999, LR: 9.996145181203615e-05


Training at 1: 100%|██████████| 452/452 [01:18<00:00,  5.74it/s]



[Generated Sample]: : : , , , , , , , ,
Before Clearing, Available memory: 18628.88 MB
Train Epoch: 1, Loss: 7.0609, LR: 9.98458666866564e-05


Training at 2: 100%|██████████| 452/452 [01:18<00:00,  5.75it/s]


...

Training at 35: 100%|██████████| 452/452 [01:31<00:00,  4.93it/s]



[Generated Sample]: KING RICHARD III : Why , I am , I
Before Clearing, Available memory: 18450.88 MB
Train Epoch: 35, Loss: 5.0006, LR: 5.782172325201153e-05


Training at 36: 100%|██████████| 452/452 [01:32<00:00,  4.86it/s]



[Generated Sample]: KING RICHARD III : Why , I am , I
Before Clearing, Available memory: 18802.88 MB
Train Epoch: 36, Loss: 4.9842, LR: 5.587686987289187e-05


Training at 37:  21%|██▏       | 97/452 [00:18<01:02,  5.68it/s]

In [41]:
inp = tok.encode("").unsqueeze(0).to(device)
print(tok.decode(model.key_value_cached_generation(inp, 50)[0].cpu()))

TypeError: Transformer.forward() got an unexpected keyword argument 'use_cache'

In [85]:
inp = tok.encode("").unsqueeze(0).to(device)
print(tok.decode(model.generate(inp, 200)[0].cpu()))

<START> springing cups immediate hell medlar ones ambitious wish't shock decorum villanous lamented torment'st subsisting scabs dad sheep-shearing trick Meantime understanding judgement windy sorrowed abodements admiral GREGORY Archbishop Victorious shelter Cambio bolder bon garden Look sworest comest coronation-day lamps anvil mingling Welcome tell'st wore shin My scabs benefactors twenty-one trident olive head-piece lodging dolours ships grumblings Am mockery Revolts Christophero merchandise Comes That vestal attended adders tacklings bating wave portal ass July steep leagues Unclasp acceptance surfeits nice usurers shapes purchasing bondslave dews car convey shines revenge knock stabbed punishment lots ride Gazed unload Slys Players friar tribe abate clogging steep regions nephew court-word visits swift canker Preoccupied Ithaca ramping associate wakened endart By churchyard lastly cedar Been wore appeals Already courage ignorant Already planted rebukes find flattery suffered beguil

In [70]:
clean_memory_cache()

Before Clearing, Available memory: 2288.92 MB


In [71]:
torch.save(model.state_dict(), "llm.pt")

In [72]:
clean_memory_cache()

Before Clearing, Available memory: 1213.69 MB


In [73]:
clean_memory_cache()

Before Clearing, Available memory: 1213.69 MB


In [74]:
inp = tok.encode("KING").unsqueeze(0).to(device)
generated = tok.decode(model.generate(inp, 200)[0].cpu())

In [108]:
inp = tok.encode("").unsqueeze(0).to(device)
tok.decode(model.generate(inp, 200)[0][1:].cpu())

"KING RICHARD III : O , I am I have you have you not have you have you . I am you , and my lord , you , you , my lord , my heart , my lord , and I have you have you , and my lord , And have you , and my lord , I am not , And not not have you , And not , I have you , And not the king , And not your own , and the king . I am not : And not the king . I have you , And not , And , And not , and the king , And , and the king , and the king 's death , and the king 's death , and the king 's son . What , and the king , And , And not , And not you have you do me , And then you , and the king , And , and the king 's the king 's death , And not , and the king , and , and the king , and the king , And , And not , ,"

In [107]:
print(generated[7:])

 KING RICHARD III : O , I am I have you have you not have you have you . I am you , and my lord , you , you , my lord , my heart , my lord , and I have you have you , and my lord , And have you , and my lord , I am not , And not not have you , And not , I have you , And not the king , And not your own , and the king . I am not : And not the king . I have you , And not , And , And not , and the king , And , and the king , and the king 's death , and the king 's death , and the king 's son . What , and the king , And , And not , And not you have you do me , And then you , and the king , And , and the king 's the king 's death , And not , and the king , and , and the king , and the king , And , And not , , and


In [95]:
x = model.generate(tok.encode("I").unsqueeze(0).to(device), 500).cpu()

In [96]:
len(x)

1

In [97]:
x

tensor([[    0,  1279,    18,  1279,  3176,  9331,    14, 11595,    14, 11595,
            14,  3222,  1279,     8, 12513, 14038,    16,  1279,  7363,  3657,
         14038,    14, 14038,    16,  1279,  3176,  9331,  3595,  2874,  8677,
            14,  1279,  3176,  2874,  8677,    14,  1279,  7363,  2874,  8677,
            14,  3222,  1279,  7363, 14038,    14,  1279,  7363,  3657, 14038,
            14,  1279,  7363,  3657, 14038,    14,  1279,  3176,  9331,  3595,
            14,  3222, 14038,    16,  1279,  7363,  3657, 14038,    14, 14038,
            14,  1279,  7363, 14038,    14,  3222,  9331,    14,  3222,  9157,
          8539,    14,  3361,  1279,  7363,  3657, 14038,    14,  3222,    14,
          3361, 14038,    14,  3222,  7363, 14038,    14,  3361, 14038,    14,
          9157,  8539,    14,  3222, 14038,  3322, 14038,    14,  3222, 14038,
            14,  3222, 14038,  7363, 14038,  7363,  3657, 14038,    14,  1279,
             8, 14038,    14,  3222, 14038,    14,  

In [105]:
tok.decode(x[0][1:])

"I : I am not , sir , sir , and I 'll tell you . I have been you , you . I am not be a man , I am a man , I have a man , and I have you , I have been you , I have been you , I am not be , and you . I have been you , you , I have you , and not , and my lord , as I have been you , and , as you , and have you , as you , my lord , and you are you , and you , and you have you have been you , I 'll you , and you , and my lord , and you , my lord , and say you , and I am not , I have you have been , and have you , and I have you are the king , and you , I 'll tell me , and you , and the world , and , that you , as you , and the king . I have you , and the world , the people , and the world , and the world , and the world , and the world , and the world , and the world , and the world , and the world , and the world , and the world , and the world , and the world , and the world , and the world , and the world , and the world , and the world , and the world , and the world , and the world , an